In [1]:
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
import gc

class PVHeadAblation:

    PREMISE_WORDS = [
        'redefine', 'define',
        'verify', 'confirm',
        'suppose', 'imagine',
        'state', 'describe'
    ]

    def __init__(self, dataset_path, model_name='gpt2', max_samples=100):
        self.dataset_path = dataset_path
        self.model_name = model_name
        self.max_samples = max_samples

        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.model = AutoModelForCausalLM.from_pretrained(model_name).to(self.device)
        self.model.eval()

        self.num_layers = len(self.model.transformer.h)
        self.num_heads = self.model.config.n_head

        self.dataset = []

    # DATA
    def load_dataset(self):
        with open(self.dataset_path, 'r') as f:
            data = json.load(f)

        if self.max_samples:
            data = data[:self.max_samples]

        for item in data:
            cf = item['base_prompt'].strip() + " " + item['target_new'] + "."
            self.dataset.append({
                'cf': cf,
                'q': item['question'],
                'factual': item['target_true'],
                'counterfactual': item['target_new']
            })

        print(f"Loaded {len(self.dataset)} samples")

    # PROMPT
    def build_prompt(self, pv, cf, q):
        return (
            f"Text: {cf}. "
            f"Question: {q}\n"
            f"Answer:"
        )

    # LOG PROB (length-normalized)
    @torch.no_grad()
    def compute_log_prob(self, prompt, answer):
        prompt_ids = self.tokenizer.encode(prompt, add_special_tokens=False)
        answer_ids = self.tokenizer.encode(answer, add_special_tokens=False)

        if not answer_ids:
            return float('-inf')

        input_ids = torch.tensor([prompt_ids + answer_ids]).to(self.device)

        outputs = self.model(input_ids)
        logits = outputs.logits[0]

        log_prob = 0.0
        for i, tok in enumerate(answer_ids):
            pos = len(prompt_ids) + i
            if pos == 0:
                continue
            probs = torch.log_softmax(logits[pos - 1], dim=-1)
            log_prob += probs[tok].item()

        return log_prob / len(answer_ids)

    # HOOK
    def get_hook(self, head_idx):
        def hook(module, input, output):
            if isinstance(output, tuple):
                attn_output = output[0]
            else:
                attn_output = output

            B, T, C = attn_output.shape
            H = module.num_heads
            head_dim = C // H

            attn_output = attn_output.view(B, T, H, head_dim)

            # ZERO ONE HEAD
            attn_output[:, :, head_idx, :] = 0

            attn_output = attn_output.view(B, T, C)

            if isinstance(output, tuple):
                return (attn_output, output[1])
            return attn_output

        return hook

    # SINGLE RUN
    def run_single(self, layer_idx, head_idx, pv):
        layer = self.model.transformer.h[layer_idx].attn
        hook = layer.register_forward_hook(self.get_hook(head_idx))

        factual = 0
        cf = 0
        deltas = []

        for item in self.dataset:
            prompt = self.build_prompt(pv, item['cf'], item['q'])

            logp_fact = self.compute_log_prob(prompt, item['factual'])
            logp_cf = self.compute_log_prob(prompt, item['counterfactual'])

            delta = logp_fact - logp_cf
            deltas.append(delta)

            if logp_fact > logp_cf:
                factual += 1
            else:
                cf += 1

        hook.remove()

        total = factual + cf

        return {
            'factual%': 100 * factual / total,
            'cf%': 100 * cf / total,
            'delta': np.mean(deltas)
        }

    # FULL RUN
    def run(self):
        self.load_dataset()

        all_results = []

        for pv in self.PREMISE_WORDS:
            print(f"\n=== Premise Word: {pv} ===")

            matrix = np.zeros((self.num_layers, self.num_heads))

            for layer in range(self.num_layers):
                for head in range(self.num_heads):
                    print(f"PV={pv} | Layer {layer} Head {head}")

                    res = self.run_single(layer, head, pv)

                    matrix[layer, head] = res['delta']

                    all_results.append({
                        'premise': pv,
                        'layer': layer,
                        'head': head,
                        'factual%': res['factual%'],
                        'cf%': res['cf%'],
                        'delta': res['delta']
                    })

                    print(f"Δ={res['delta']:.4f}, Factual%={res['factual%']:.2f}")

                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                    gc.collect()

            self.plot_heatmap(matrix, pv)

        self.save_results(all_results)

    # PLOT
    def plot_heatmap(self, matrix, pv):
        plt.figure(figsize=(10, 5))
        plt.imshow(matrix, aspect='auto')
        plt.colorbar(label="Δ")
        plt.xlabel("Head")
        plt.ylabel("Layer")
        plt.title(f"Head Ablation Heatmap (PV = {pv})")
        plt.savefig(f"heatmap_{pv}.png", dpi=150)
        plt.close()

    # SAVE
    def save_results(self, results):
        df = pd.DataFrame(results)
        df.to_csv("pv_head_ablation_results.csv", index=False)
        print("\nSaved results to pv_head_ablation_results.csv")


# RUN
if __name__ == "__main__":
    exp = PVHeadAblation(
        dataset_path="../../../Data/gpt2_with_questions_merged.json",
        model_name="gpt2",
        max_samples=500
    )
    exp.run()

/home/animesh-lohar-2711/anaconda3/envs/comp_mech_gpu/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/animesh-lohar-2711/anaconda3/envs/comp_mech_gpu/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Loaded 500 samples

=== Premise Word: redefine ===
PV=redefine | Layer 0 Head 0
Δ=0.8852, Factual%=72.00
PV=redefine | Layer 0 Head 1
Δ=0.7490, Factual%=69.80
PV=redefine | Layer 0 Head 2
Δ=0.8946, Factual%=72.20
PV=redefine | Layer 0 Head 3
Δ=0.8253, Factual%=71.60
PV=redefine | Layer 0 Head 4
Δ=0.7474, Factual%=70.40
PV=redefine | Layer 0 Head 5
Δ=0.7450, Factual%=71.20
PV=redefine | Layer 0 Head 6
Δ=0.7759, Factual%=71.20
PV=redefine | Layer 0 Head 7
Δ=0.8503, Factual%=72.20
PV=redefine | Layer 0 Head 8
Δ=0.8294, Factual%=71.00
PV=redefine | Layer 0 Head 9
Δ=0.8541, Factual%=72.20
PV=redefine | Layer 0 Head 10
Δ=0.8455, Factual%=72.00
PV=redefine | Layer 0 Head 11
Δ=0.7700, Factual%=70.60
PV=redefine | Layer 1 Head 0
Δ=0.8380, Factual%=71.80
PV=redefine | Layer 1 Head 1
Δ=0.7880, Factual%=71.00
PV=redefine | Layer 1 Head 2
Δ=0.8401, Factual%=71.80
PV=redefine | Layer 1 Head 3
Δ=0.8306, Factual%=71.60
PV=redefine | Layer 1 Head 4
Δ=0.7914, Factual%=71.40
PV=redefine | Layer 1 Head 5
